In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import math
import re
import pandas as pd
import jax.numpy as jnp
from jax.typing import ArrayLike

import altair as alt

from mixres.sim import PoissonDisjoint1D

## Utility Functions

In [3]:
_INTERVAL_PATTERN = re.compile(
    r"^\s*([\[(])\s*([-+]?\d+(?:\.\d+)?)\s*,\s*([-+]?\d+(?:\.\d+)?)\s*([\])])\s*$"
)


def expand_interval_dataframe(
    df: pd.DataFrame,
    interval_col: str | None = None,
    x_col: str = "x",
) -> pd.DataFrame:
    """Expand interval-indexed rows into one row per integer point.

    Parameters
    ----------
    df : pandas.DataFrame
        Input data containing an interval column plus any columns to copy.
    interval_col : str or None, optional
        Name of the interval column. If None, the function will try to infer it
        from string columns containing values like "[0,5)" or "[95, 100]".
    x_col : str, optional
        Name of the integer-valued output column. Defaults to "x".

    Returns
    -------
    pandas.DataFrame
        Expanded dataframe with one row per integer in each interval.
    """
    if df.empty:
        out = df.copy()
        out[x_col] = pd.Series(dtype="int64")
        return out

    working_df = df.copy()

    if interval_col is None:
        candidate_cols = []
        for col in working_df.columns:
            series = working_df[col].dropna().astype(str)
            if not series.empty and series.str.match(_INTERVAL_PATTERN).all():
                candidate_cols.append(col)

        if not candidate_cols:
            raise ValueError(
                "Could not infer the interval column. Pass interval_col explicitly."
            )
        interval_col = candidate_cols[0]

    expanded_rows = []

    for _, row in working_df.iterrows():
        interval_value = row[interval_col]

        if isinstance(interval_value, pd.Interval):
            left = float(interval_value.left)
            right = float(interval_value.right)
            left_closed = interval_value.closed in {"left", "both"}
            right_closed = interval_value.closed in {"right", "both"}
        else:
            match = _INTERVAL_PATTERN.match(str(interval_value))
            if match is None:
                raise ValueError(
                    f"Could not parse interval {interval_value!r} in column {interval_col!r}."
                )

            left_bracket, left_str, right_str, right_bracket = match.groups()
            left = float(left_str)
            right = float(right_str)
            left_closed = left_bracket == "["
            right_closed = right_bracket == "]"

        start = math.ceil(left) if left_closed else math.floor(left) + 1
        end = math.floor(right) if right_closed else math.ceil(right) - 1

        for x in range(start, end + 1):
            new_row = row.to_dict()
            new_row[x_col] = x
            expanded_rows.append(new_row)

    return pd.DataFrame(expanded_rows).reset_index(drop=True)


def compute_interval_widths(intervals: list[str], dtype=np.float32) -> np.ndarray:
    """Compute integer widths for interval strings like "[0,5)" or "[95,100]"."""
    if not intervals:
        raise ValueError("intervals must contain at least one interval string.")

    widths = []
    for interval in intervals:
        match = _INTERVAL_PATTERN.match(str(interval))
        if match is None:
            raise ValueError(f"Could not parse interval: {interval!r}")

        left_bracket, left_str, right_str, right_bracket = match.groups()
        left = float(left_str)
        right = float(right_str)

        start = math.ceil(left) if left_bracket == "[" else math.floor(left) + 1
        end = math.floor(right) if right_bracket == "]" else math.ceil(right) - 1

        if end < start:
            raise ValueError(f"Interval contains no integer points: {interval!r}")

        widths.append(end - start + 1)

    return np.asarray(widths, dtype=dtype)


def build_interval_sum_matrix(
    intervals: list[str],
    dtype=jnp.float32,
    return_support: bool = False,
):
    """Build a JAX matrix that sums a latent vector over integer intervals.

    Each row corresponds to one interval string such as "[0,5)" or "[95, 100]".
    The columns correspond to the integer support from the smallest included
    integer to the largest included integer. Entries are 1 inside the interval
    and 0 elsewhere.
    """
    if not intervals:
        raise ValueError("intervals must contain at least one interval string.")

    parsed_intervals = []
    min_x = np.inf
    max_x = -np.inf

    for interval in intervals:
        match = _INTERVAL_PATTERN.match(str(interval))
        if match is None:
            raise ValueError(f"Could not parse interval: {interval!r}")

        left_bracket, left_str, right_str, right_bracket = match.groups()
        left = float(left_str)
        right = float(right_str)

        start = math.ceil(left) if left_bracket == "[" else math.floor(left) + 1
        end = math.floor(right) if right_bracket == "]" else math.ceil(right) - 1

        if end < start:
            raise ValueError(f"Interval contains no integer points: {interval!r}")

        parsed_intervals.append((start, end))
        min_x = min(min_x, start)
        max_x = max(max_x, end)

    width = int(max_x - min_x + 1)
    matrix = np.zeros((len(parsed_intervals), width), dtype=np.float32)

    for row_idx, (start, end) in enumerate(parsed_intervals):
        matrix[row_idx, int(start - min_x) : int(end - min_x + 1)] = 1.0

    matrix = jnp.asarray(matrix, dtype=dtype)

    if return_support:
        support = jnp.arange(int(min_x), int(max_x) + 1)
        return matrix, support

    return matrix

## Analysis

In [4]:
# Generate the dataset
df = PoissonDisjoint1D(cut_points=[18, 64], n_observations=100, seed=42).generate()
df_expanded = expand_interval_dataframe(df, interval_col="interval", x_col="x")

## Modeling

In [5]:
df_model = df.groupby(["interval_id"])["y"].agg(N="size", y="sum").reset_index()

In [6]:
x = np.arange(0, 101)
y_obs = df_model["y"].values
interval_id = df_model["interval_id"].values
N = df_model["N"].values

intervals = df["interval"].unique().tolist()
A = build_interval_sum_matrix(intervals)
interval_width = compute_interval_widths(intervals)

# Map each x point to its segment index (0=>[0,17], 1=>[18,63], 2=>[64,100])
segment_id_x = np.select([x < 18, x < 64], [0, 1], default=2).astype(np.int32)
n_segments = int(segment_id_x.max() + 1)

# Per-segment HSGP precomputations
cut_points_hsgp = [18, 64]
x_segs = [
    x[x < cut_points_hsgp[0]],
    x[(x >= cut_points_hsgp[0]) & (x < cut_points_hsgp[1])],
    x[x >= cut_points_hsgp[1]],
]
# Standardize each segment (centre then divide by std) so HSGP inputs are unit-scale
x_segs_standardized = [(xs - xs.mean()) / xs.std() for xs in x_segs]
# ell: 1.5 × max|x_std| so the domain boundary is comfortably outside the data range
ell_segs = [float(1.5 * np.abs(xs).max()) for xs in x_segs_standardized]
# m: fixed number of basis functions per segment
m_segs = [15] * n_segments

print(f"Segment lengths : {[len(xs) for xs in x_segs]}")
print(f"ell_segs        : {[round(e, 3) for e in ell_segs]}")
print(f"m_segs          : {m_segs}")

# Convert to JAX arrays
y_obs = jnp.array(y_obs)
interval_id = jnp.array(interval_id)
N = jnp.array(N)
A_jax = jnp.array(A)
interval_width_jax = jnp.array(interval_width)
segment_id_x_jax = jnp.array(segment_id_x)
x_segs_jax = [jnp.array(xs.astype(np.float32)) for xs in x_segs_standardized]

Segment lengths : [18, 46, 37]
ell_segs        : [2.458, 2.542, 2.529]
m_segs          : [15, 15, 15]


In [7]:
import jax
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist
from numpyro.infer import autoguide
from numpyro.contrib.hsgp.approximation import hsgp_squared_exponential

from mixres.models._inference import (
    run_inference_mcmc,
    run_inference_svi,
    posterior_predictive_mcmc,
    posterior_predictive_svi,
)

/Users/shozen/Imperial/0_Research/mixres/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
def model_hsgp(
    A: ArrayLike,
    x_segs: list,
    ell_segs: list[float],
    m_segs: list[int],
    N: ArrayLike,
    interval_id: ArrayLike,
    interval_width: ArrayLike,
    segment_id_x: ArrayLike,
    n_segments: int,
    y_obs: ArrayLike = None,
):
    """
    Bayesian Poisson model with per-segment HSGP (squared exponential kernel)
    and per-segment intercepts.

    Each disjoint interval gets its own independent GP with independent
    amplitude and length-scale hyperparameters. The GP is evaluated on
    standardized x values so that each segment lies within [-ell, +ell].

    For each segment s:
        amplitude_s ~ Gamma(5, 5)     GP amplitude
        length_s    ~ Gamma(5, 25)    GP length scale in standardised x units
        f_s = HSGP_SE(x_s; amplitude_s, length_s, ell_s, m_s)   [non-centred]

    Global and per-segment intercepts:
        beta0 ~ N(0, 1)
        alpha ~ N(0, 1)^n_segments

    Likelihood:
        lam = exp(beta0 + alpha[segment_id_x] + concat(f_segs))
        y ~ Poisson(A @ lam / interval_width * N[interval_id])
    """
    # --- Intercepts ---
    beta0 = numpyro.sample("beta0", dist.Normal(0, 1))
    alpha = numpyro.sample("alpha", dist.Normal(0, 1).expand([n_segments]).to_event(1))

    # --- Per-segment GP components ---
    # numpyro.handlers.scope prefixes all internal sample sites (e.g. "beta") so
    # that three independent calls to hsgp_squared_exponential don't collide.
    f_segs = []
    for seg_idx in range(n_segments):
        amplitude = numpyro.sample(
            f"amplitude_{seg_idx}",
            dist.Gamma(5.0, 5.0),
        )
        length = numpyro.sample(
            f"length_{seg_idx}",
            dist.Gamma(5.0, 25.0),
        )
        with numpyro.handlers.scope(prefix=f"gp_{seg_idx}", divider="/"):
            f_seg = hsgp_squared_exponential(
                x=x_segs[seg_idx],
                alpha=amplitude,
                length=length,
                ell=ell_segs[seg_idx],
                m=m_segs[seg_idx],
                non_centered=True,
            )
        f_segs.append(f_seg)

    # Concatenate segment GPs; x = [0,...,100] is sorted so segments are contiguous
    f = jnp.concatenate(f_segs)  # shape (101,)

    # --- Latent rate ---
    lam = numpyro.deterministic("lam", jnp.exp(beta0 + alpha[segment_id_x] + f))
    lam_interval = (
        jnp.matmul(A, lam) / interval_width * N[interval_id]
    )  # Average rate over each interval (divide by interval-specific width)

    # --- Likelihood ---
    # Use interval_id.shape[0] instead of len(y_obs) so the plate size is available
    # both during inference (y_obs provided) and prediction (y_obs=None).
    with numpyro.plate("obs", interval_id.shape[0]):
        numpyro.sample("y", dist.Poisson(lam_interval[interval_id]), obs=y_obs)

In [9]:
# MCMC inference with NUTS
rng_key_mcmc = jax.random.PRNGKey(0)
mcmc = run_inference_mcmc(
    rng_key_mcmc,
    model_hsgp,
    A=A_jax,
    x_segs=x_segs_jax,
    ell_segs=ell_segs,
    m_segs=m_segs,
    N=N,
    interval_id=interval_id,
    interval_width=interval_width_jax,
    segment_id_x=segment_id_x_jax,
    n_segments=n_segments,
    y_obs=y_obs,
)
mcmc_samples = mcmc.get_samples()
print(mcmc_samples.keys())

/Users/shozen/Imperial/0_Research/mixres/mixres/models/_inference.py:28: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  mcmc = MCMC(
sample: 100%|██████████| 1000/1000 [00:01<00:00, 583.00it/s, 63 steps of size 6.44e-02. acc. prob=0.91]


Number of divergences: 0
dict_keys(['alpha', 'amplitude_0', 'amplitude_1', 'amplitude_2', 'beta0', 'gp_0/beta', 'gp_1/beta', 'gp_2/beta', 'lam', 'length_0', 'length_1', 'length_2'])


In [10]:
# Extract posterior samples of lam from MCMC
lam = mcmc_samples["lam"].mean(axis=0)
lam_lower = jnp.percentile(mcmc_samples["lam"], 2.5, axis=0)
lam_upper = jnp.percentile(mcmc_samples["lam"], 97.5, axis=0)

df_results = df_expanded[["x", "lambda"]].drop_duplicates().reset_index(drop=True)
df_results["lam"] = lam
df_results["lam_lower"] = lam_lower
df_results["lam_upper"] = lam_upper
df_results["segment"] = np.select(
    [df_results["x"] < 18, df_results["x"] < 64],
    [0, 1],
    default=2,
)

# detail="segment:N" draws three discontinuous lines, one per disjoint interval
true_line = (
    alt.Chart(df_results)
    .mark_line(color="red")
    .encode(x="x", y="lambda", detail="segment:N")
)
mean_line = (
    alt.Chart(df_results)
    .mark_line(color="blue")
    .encode(x="x", y="lam", detail="segment:N")
)
bands = (
    alt.Chart(df_results)
    .mark_area(color="lightblue", opacity=0.5)
    .encode(
        x="x",
        y="lam_lower:Q",
        y2="lam_upper:Q",
        detail="segment:N",
    )
)
true_line + mean_line + bands

alt.LayerChart(...)

In [15]:
# SVI with AutoNormal guide
rng_key_svi = jax.random.PRNGKey(1)
guide_hsgp = autoguide.AutoNormal(model_hsgp)
svi_result = run_inference_svi(
    rng_key_svi,
    model_hsgp,
    guide_hsgp,
    num_steps=20_000,
    A=A_jax,
    x_segs=x_segs_jax,
    ell_segs=ell_segs,
    m_segs=m_segs,
    N=N,
    interval_id=interval_id,
    interval_width=interval_width_jax,
    segment_id_x=segment_id_x_jax,
    n_segments=n_segments,
    y_obs=y_obs,
)

100%|██████████| 20000/20000 [00:03<00:00, 5530.00it/s, init loss: 459.7531, avg. loss [19001-20000]: 28.1675]


In [17]:
# Draw posterior predictive samples from the fitted SVI guide
rng_key_pp = jax.random.PRNGKey(2)
svi_samples = posterior_predictive_svi(
    rng_key_pp,
    model_hsgp,
    guide_hsgp,
    svi_result.params,
    A=A_jax,
    x_segs=x_segs_jax,
    ell_segs=ell_segs,
    m_segs=m_segs,
    N=N,
    interval_id=interval_id,
    interval_width=interval_width_jax,
    segment_id_x=segment_id_x_jax,
    n_segments=n_segments,
)
print(svi_samples.keys())

dict_keys(['lam', 'y'])


In [18]:
# Extract posterior samples of lam from SVI
lam = jnp.array(svi_samples["lam"]).mean(axis=0)
lam_lower = jnp.percentile(svi_samples["lam"], 2.5, axis=0)
lam_upper = jnp.percentile(svi_samples["lam"], 97.5, axis=0)

df_results = df_expanded[["x", "lambda"]].drop_duplicates().reset_index(drop=True)
df_results["lam"] = lam
df_results["lam_lower"] = lam_lower
df_results["lam_upper"] = lam_upper
df_results["segment"] = np.select(
    [df_results["x"] < 18, df_results["x"] < 64],
    [0, 1],
    default=2,
)

# detail="segment:N" draws three discontinuous lines, one per disjoint interval
true_line = (
    alt.Chart(df_results)
    .mark_line(color="red")
    .encode(x="x", y="lambda", detail="segment:N")
)
mean_line = (
    alt.Chart(df_results)
    .mark_line(color="blue")
    .encode(x="x", y="lam", detail="segment:N")
)
bands = (
    alt.Chart(df_results)
    .mark_area(color="lightblue", opacity=0.5)
    .encode(
        x="x",
        y="lam_lower:Q",
        y2="lam_upper:Q",
        detail="segment:N",
    )
)
true_line + mean_line + bands

alt.LayerChart(...)